In [9]:
import os

In [2]:
%pwd

'd:\\ML-Projects\\End-to-end-Machine-Learning-World-Development-Measurement-Clustering-Analysis-with-MLFlow\\notebooks'

In [3]:
os.chdir('../')

In [4]:
%pwd

'd:\\ML-Projects\\End-to-end-Machine-Learning-World-Development-Measurement-Clustering-Analysis-with-MLFlow'

In [5]:
# PReparing Data Ingestion entity and config
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [17]:
from wdmproject.constants import *
from wdmproject.utils.common import read_yaml, create_directories

In [18]:
## Configuration manager class
class ConfigurationManager:
    def __init__(
        self,
        config_filepath: Path = CONFIG_FILE_PATH,
        params_filepath: Path = PARAMS_FILE_PATH,
        schema_filepath: Path = SCHEMA_FILE_PATH
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)
        
        create_directories([self.config.artifacts_root])
    
    # Data Ingestion related configuration

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion
        
        create_directories([config.root_dir])
        
        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir
        )
        
        return data_ingestion_config

In [21]:
## Components of Data Ingestion
import os
import urllib.request as request
import zipfile
from wdmproject.utils.common import save_json, load_json, save_bin, load_bin, get_size
from wdmproject import logger

In [25]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url=self.config.source_URL,
                filename=self.config.local_data_file
            )
            logger.info(f"file downloaded successfully and saved at: {filename}\n{headers}")
            logger.info(f"file size: {get_size(Path(filename))}")
        else:
            logger.info(f"file already exists of size: {get_size(Path(self.config.local_data_file))}")
            logger.info(f"file already exists at: {self.config.local_data_file}")

    def extract_zip_file(self):
        """
        zip_file_path: str 
        Extracts the zip file into the data directory
        Function returns None
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)
        logger.info(f"file extracted successfully at: {self.config.unzip_dir}")

In [26]:
# Create a pipeline of data ingestion
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    logger.exception(e)

[2026-02-28 16:58:18,390: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-02-28 16:58:18,394: INFO: common: yaml file: params.yaml loaded successfully]
[2026-02-28 16:58:18,397: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-02-28 16:58:18,400: INFO: common: created directory at: artifacts]
[2026-02-28 16:58:18,405: INFO: common: created directory at: artifacts/data_ingestion]
[2026-02-28 16:58:18,408: INFO: 290105069: file already exists of size: ~ 354 KB]
[2026-02-28 16:58:18,410: INFO: 290105069: file already exists at: artifacts/data_ingestion/data.zip]
[2026-02-28 16:58:18,422: INFO: 290105069: file extracted successfully at: artifacts/data_ingestion]
